In [ ]:
from google.colab import files
Uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_csv("Student Social Media And Mental Health Impact.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df.shape

In [ ]:
df.head(3)

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
sns.histplot(df['Mental_Health_Score'],kde=True)

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=True)

In [ ]:
df['Stress_Level'].unique()

In [ ]:
order =['Medium', 'Low', 'Very High', 'High']
sns.boxplot(x='Stress_Level',y='Mental_Health_Score',data=df,order = order)

In [ ]:
sns.scatterplot(x='Avg_Daily_Usage_Hours',y='Mental_Health_Score',data=df)

In [ ]:
df.columns

In [ ]:
sns.scatterplot(x='Sleep_Hours_Per_Night',y = 'Mental_Health_Score',data=df)

In [ ]:
sns.countplot(x=df['Most_Used_Platform'],order=df['Most_Used_Platform'].value_counts().index)

In [ ]:
df['Most_Used_Platform'].value_counts()

In [ ]:
num_features = df.select_dtypes(include='number')
Q1 = num_features.quantile(0.25)
Q3 = num_features.quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

outliers = ((num_features < lower_bound) | (num_features > upper_bound)).any(axis=1)
print(outliers.sum())

In [ ]:
df = df.drop_duplicates()

In [ ]:
df['Physical_Activity_Hours']=df['Physical_Activity_Hours'].clip(lower=0)

In [ ]:
df.describe()

In [ ]:
num_features = df.select_dtypes(include='number')
num_features.skew()

In [ ]:
df['Country'].value_counts().index[:10].tolist()

In [ ]:
top_countries = df['Country'].value_counts().index[:10].tolist()

In [ ]:
def group_countries(country):
  if country in top_countries:
    return country
  else:
    return 'Other'

In [ ]:
df['Grouped_country'] = df['Country'].apply(group_countries)

In [ ]:
df['Grouped_country'].value_counts()

In [ ]:
print(df.columns.tolist())

In [ ]:
X = df.drop('Mental_Health_Score', axis=1)
y = df['Mental_Health_Score']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30,random_state=42)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30,random_state=42)

skwewd_col = ["Study_Hours"]

other_numeric_cols = [
    "Age",
    "Avg_Daily_Usage_Hours",
    "Daily_Unlocks",
    "Physical_Activity_Hours",
    "Sleep_Hours_Per_Night"
]

ordinal_col = ["Stress_Level"]

normal_col = [
    "Gender",
    "Academic_Level",
    "Most_Used_Platform",
    "Purpose_Of_Use",
    "Grouped_country"
]

feature_col = skwewd_col + other_numeric_cols + ordinal_col + normal_col

X = df[feature_col]

y = df["Mental_Health_Score"]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

#1 Skewed features
skew_pipeline = Pipeline(steps=[
    ('log_transform',FunctionTransformer(np.log1p)),
    ('scale',StandardScaler())
])
#2 Numeric Features
plain_numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])
#3 Ordinal
ordinal_pipeline = Pipeline(steps=[
    ('encode',OrdinalEncoder(categories=[['Low','Medium','High','Very High']]))

])
#4 Nominal Features
nominal_pipeline = Pipeline(steps=[
    ('encode',OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('Skewed_Pipeline',skew_pipeline,skwewd_col),
    ('Plain_Numeric',plain_numeric_pipeline,other_numeric_cols),
    ('Ordinal',ordinal_pipeline,ordinal_col),
    ('Nominal',nominal_pipeline,normal_col)
])


In [ ]:
print(preprocessor)

In [ ]:
print(X_train.columns)

In [ ]:
print(X_train.columns.tolist())

In [ ]:
print(df.columns.tolist())

In [ ]:
'Daily_unlocks'

In [ ]:
numerical_features = [
    'Daily_Unlocks',
    ...
]

In [ ]:
preprocessor = ColumnTransformer([
    ('num',StandardScaler(),numerical_features),
    ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_features)
])

In [ ]:
categorical_features = [
    'Gender',
    'City',
    'Stress_Level'
]

In [ ]:
numerical_features = [
    'Age',
    'Daily_Unlocks',
]

categorical_features = [
    'Gender',
    'Stress_Level'
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

In [ ]:
print(type(numerical_features))
print(numerical_features)

print(type(categorical_features))
print(categorical_features)

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

In [ ]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
 lr_pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
from sklearn.pipeline import Pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])
lr_pipeline.fit(X_train , y_train)
lr_pred =lr_pipeline.predict(X_test)

lr_pred_train = lr_pipeline.predict(X_train)

lr_r2_testing  = r2_score(y_test,lr_pred)
lr_r2_training = r2_score(y_train,lr_pred_train)
lr_mean_absolute_error = mean_absolute_error(y_test,lr_pred)

print(f"Accuracy of Training {lr_r2_training}")
print(f"Accuracy of Testing {lr_r2_testing}")
print(f"  Mean Absolute Error {lr_mean_absolute_error}")


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('random forest',RandomForestRegressor(random_state=42))
])
rf_pipeline.fit(X_train,y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_pred_train = rf_pipeline.predict(X_train)

rf_r2_testing = r2_score(y_test,rf_pred)
rf_r2_training = r2_score(y_train,rf_pred_train)
rf_mean_absolute_error = mean_absolute_error(y_test,rf_pred)

print(f"Accuracy of Training {rf_r2_training}")
print(f"Accuracy of Testing {rf_r2_testing}")
print(f"  Mean Absolute Error {rf_mean_absolute_error}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    'random forest__n_estimators':[100,200,300],
    'random forest__max_depth':[5,10,15],
    'random forest__min_samples_split':[2,5,10],
    'random forest__min_samples_leaf':[1,2,4]
}
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=15,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train,y_train)


In [ ]:
random_search.best_params_
{'random forest__max_depth': 15,
 'random forest__min_samples_leaf': 2,
 'random forest__min_samples_split': 5,
 'random forest__n_estimators': 200}



In [ ]:
rf_best_pipeline  = random_search.best_estimator_
rf_best_pred = rf_best_pipeline.predict(X_test)

print(f"R2 Score for random forest after hyperparameter tuning:{r2_score(y_test,rf_best_pred)}")
print(f"MAE Score for random forest after hyperparameter tuning:{mean_absolute_error(y_test,rf_best_pred)}")

In [ ]:
from sklearn.metrics import mean_squared_error
lr_rmse = np.sqrt(mean_squared_error(y_test,lr_pred))
rf_rmse = np.sqrt(mean_squared_error(y_test,rf_pred))

rf_tuned_pred = random_search.best_estimator_.predict(X_test)
rf_tuned_rmse = np.sqrt(mean_squared_error(y_test,rf_tuned_pred))
rf_tuned_training_pred = random_search.best_estimator_.predict(X_train)
rf_tuned_training = r2_score(y_train,rf_tuned_training_pred)
rf_tuned_mae = mean_absolute_error(y_test,rf_tuned_pred)
rf_tuned_r2 = r2_score(y_test,rf_tuned_pred)

results = pd.DataFrame({
    'Model':['Linear Regression','Random Forest','Random Forest Tuned'],
    'R2':[lr_r2_testing , rf_r2_testing,rf_tuned_r2],
    'Trainig R2':[lr_r2_training,rf_r2_training,rf_tuned_training],
    'RMSE':[lr_rmse,rf_rmse,rf_tuned_rmse],
    'MAE':[lr_mean_absolute_error,rf_mean_absolute_error,rf_tuned_mae]

})
print(results)

In [ ]:
import joblib
joblib.dump(rf_pipeline,'Mental_Health_Model.pkl')